In [ ]:
%pyspark
# ================================================================
# Germany Air Pollution Analysis 2023
# Dataset: SDS011 sensor network
# P1 = PM10, P2 = PM2.5
# Paste this whole block into one Zeppelin paragraph and run it.
# ================================================================

from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType, TimestampType
)
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.animation as animation
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.patches as mpatches
import warnings

warnings.filterwarnings("ignore")
spark.catalog.clearCache()

plt.rcParams.update({
    "figure.dpi": 120,
    "figure.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

WHO = {"PM10": 45, "PM25": 15}
COL = {"PM10": "#D94F3D", "PM25": "#3D7ED9"}

schema = StructType([
    StructField("sensor_id", IntegerType(), True),
    StructField("sensor_type", StringType(), True),
    StructField("location", IntegerType(), True),
    StructField("lat", DoubleType(), True),
    StructField("lon", DoubleType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("P1", DoubleType(), True),
    StructField("durP1", DoubleType(), True),
    StructField("ratioP1", DoubleType(), True),
    StructField("P2", DoubleType(), True),
    StructField("durP2", DoubleType(), True),
    StructField("ratioP2", DoubleType(), True),
])

# ================================================================
# 1. Load and clean
# No cache() and no count(): avoids "No space left on device" errors.
# ================================================================

raw = (
    spark.read
    .option("header", "true")
    .option("delimiter", ";")
    .schema(schema)
    .csv("hdfs://bdgtm:8020/pm_data/DE_2023-*_sds011.csv")
)

clean = (
    raw.select("sensor_id", "lat", "lon", "timestamp", "P1", "P2")
    .filter(F.col("timestamp").isNotNull())
    .filter(F.col("P1").isNotNull() & F.col("P2").isNotNull())
    .filter(F.col("P1").between(0, 500) & F.col("P2").between(0, 500))
    .withColumn("date", F.to_date("timestamp"))
    .withColumn("hour", F.hour("timestamp"))
    .withColumn("day_of_week", F.dayofweek("timestamp"))
    .withColumn("month", F.month("timestamp"))
)

print("Clean data ready. Small preview:")
clean.limit(3).show(truncate=False)

# ================================================================
# 2. Daily trend and moving averages
# ================================================================

daily = (
    clean.groupBy("date")
    .agg(F.avg("P1").alias("PM10"), F.avg("P2").alias("PM25"))
    .orderBy("date")
    .toPandas()
)

if daily.empty:
    raise ValueError("No rows were loaded. Check the HDFS path and date files.")

daily["date"] = pd.to_datetime(daily["date"])
daily = daily.sort_values("date").set_index("date")
daily["PM10_7d"] = daily["PM10"].rolling(7, min_periods=3).mean()
daily["PM25_7d"] = daily["PM25"].rolling(7, min_periods=3).mean()
daily["PM10_30d"] = daily["PM10"].rolling(30, min_periods=7).mean()
daily["PM25_30d"] = daily["PM25"].rolling(30, min_periods=7).mean()

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
fig.suptitle("Daily PM Concentrations with Moving Averages - Germany 2023",
             fontsize=13, fontweight="bold")

for ax, metric in zip(axes, ["PM10", "PM25"]):
    ax.plot(daily.index, daily[metric], color=COL[metric], lw=0.9, alpha=0.55,
            label="Daily mean")
    ax.plot(daily.index, daily[metric + "_7d"], color="darkorange", lw=1.6,
            label="7-day moving average")
    ax.plot(daily.index, daily[metric + "_30d"], color="black", lw=1.6, ls="--",
            label="30-day moving average")
    ax.axhline(WHO[metric], color="red", ls=":", lw=1.3,
               label="WHO limit (" + str(WHO[metric]) + " ug/m3)")
    ax.set_ylabel(metric + " (ug/m3)")
    ax.legend(fontsize=8, ncol=4)
    ax.grid(alpha=0.25)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))

plt.tight_layout()
plt.savefig("step1_daily_moving_average.png", bbox_inches="tight")
plt.show()
print("Saved: step1_daily_moving_average.png")

# Python-only dynamic output: animated PM2.5 timeline.
# This is lightweight because it animates the daily summary, not raw Spark rows.
try:
    pm25_anim = daily["PM25"].dropna()
    frame_count = min(90, len(pm25_anim))
    frame_idx = np.linspace(3, len(pm25_anim) - 1, frame_count).astype(int)

    fig, ax = plt.subplots(figsize=(12, 5))
    fig.suptitle("Animated PM2.5 Timeline - Germany 2023",
                 fontsize=14, fontweight="bold")
    ax.set_facecolor("#f8fafc")
    ax.set_xlim(pm25_anim.index.min(), pm25_anim.index.max())
    ax.set_ylim(0, max(float(pm25_anim.max()) * 1.18, WHO["PM25"] * 1.25))
    ax.set_ylabel("PM2.5 (ug/m3)")
    ax.grid(alpha=0.25)
    ax.axhline(WHO["PM25"], color="red", ls=":", lw=1.5, label="WHO limit")
    line, = ax.plot([], [], color=COL["PM25"], lw=2.5, label="Daily PM2.5")
    dot, = ax.plot([], [], "o", color=COL["PM25"], ms=7)
    label = ax.text(0.02, 0.90, "", transform=ax.transAxes,
                    fontsize=12, fontweight="bold",
                    bbox=dict(facecolor="white", edgecolor="#d9e2ec", alpha=0.9))
    ax.legend(loc="upper right")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))

    def update_pm25(frame):
        upto = frame_idx[frame]
        x = pm25_anim.index[:upto + 1]
        y = pm25_anim.values[:upto + 1]
        line.set_data(x, y)
        dot.set_data([x[-1]], [y[-1]])
        label.set_text(str(x[-1].date()) + " | PM2.5 = " +
                       str(round(float(y[-1]), 2)) + " ug/m3")
        return line, dot, label

    ani = animation.FuncAnimation(fig, update_pm25, frames=len(frame_idx),
                                  interval=90, blit=True)
    ani.save("step1b_pm25_animated_timeline.gif",
             writer=animation.PillowWriter(fps=12))
    plt.close(fig)
    print("Saved: step1b_pm25_animated_timeline.gif")
except Exception as e:
    print("Animation skipped:", e)

# ================================================================
# 3. Seasonal patterns: hour, weekday, month
# ================================================================

hourly = (
    clean.groupBy("hour")
    .agg(F.avg("P1").alias("PM10"), F.avg("P2").alias("PM25"))
    .orderBy("hour")
    .toPandas()
)

dow = (
    clean.groupBy("day_of_week")
    .agg(F.avg("P1").alias("PM10"), F.avg("P2").alias("PM25"))
    .orderBy("day_of_week")
    .toPandas()
)
dow["label"] = dow["day_of_week"].map({
    1: "Sun", 2: "Mon", 3: "Tue", 4: "Wed", 5: "Thu", 6: "Fri", 7: "Sat"
})

monthly = (
    clean.groupBy("month")
    .agg(F.avg("P1").alias("PM10"), F.avg("P2").alias("PM25"))
    .orderBy("month")
    .toPandas()
)
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Seasonal Patterns - PM10 and PM2.5", fontsize=13, fontweight="bold")

axes[0].plot(hourly["hour"], hourly["PM10"], marker="o", color=COL["PM10"], label="PM10")
axes[0].plot(hourly["hour"], hourly["PM25"], marker="s", color=COL["PM25"], label="PM2.5")
axes[0].set_title("Hour of Day")
axes[0].set_xlabel("Hour")
axes[0].set_ylabel("ug/m3")
axes[0].set_xticks(range(0, 24, 3))
axes[0].legend(fontsize=8)

x = range(len(dow))
axes[1].bar(x, dow["PM10"], color=COL["PM10"], alpha=0.75, label="PM10")
axes[1].bar(x, dow["PM25"], color=COL["PM25"], alpha=0.75, label="PM2.5")
axes[1].set_title("Day of Week")
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(dow["label"])
axes[1].legend(fontsize=8)

mx = range(len(monthly))
axes[2].plot(mx, monthly["PM10"], marker="o", color=COL["PM10"], label="PM10")
axes[2].plot(mx, monthly["PM25"], marker="s", color=COL["PM25"], label="PM2.5")
axes[2].set_title("Month")
axes[2].set_xticks(list(mx))
axes[2].set_xticklabels([month_labels[int(m) - 1] for m in monthly["month"]], rotation=45)
axes[2].legend(fontsize=8)

for ax in axes:
    ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.savefig("step2_seasonal_patterns.png", bbox_inches="tight")
plt.show()
print("Saved: step2_seasonal_patterns.png")

# ================================================================
# 4. Outlier detection
# Main method: IQR. Comparison method: z-score.
#
# Presentation notes:
# - Point outlier: one unusual value/day.
# - Contextual outlier: unusual only for its context/season/time.
# - Collective outlier: a group of values unusual together.
# - Other methods: z-score, rolling z-score, DBSCAN, Isolation Forest.
# - If outliers are not removed: winsorise, cap, log-transform, or use robust models.
# ================================================================

def flag_iqr(series, k=1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lo = q1 - k * iqr
    hi = q3 + k * iqr
    return (series < lo) | (series > hi), lo, hi

def flag_zscore(series, threshold=3.0):
    sd = series.std()
    if sd == 0 or pd.isna(sd):
        return pd.Series(False, index=series.index)
    z = (series - series.mean()) / sd
    return z.abs() > threshold

daily["PM10_iqr"], pm10_lo, pm10_hi = flag_iqr(daily["PM10"])
daily["PM25_iqr"], pm25_lo, pm25_hi = flag_iqr(daily["PM25"])
daily["PM10_z"] = flag_zscore(daily["PM10"])
daily["PM25_z"] = flag_zscore(daily["PM25"])

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
fig.suptitle("Outlier Detection with IQR - Germany 2023",
             fontsize=13, fontweight="bold")

for ax, metric, lo, hi in zip(axes, ["PM10", "PM25"], [pm10_lo, pm25_lo], [pm10_hi, pm25_hi]):
    flag = metric + "_iqr"
    ax.plot(daily.index, daily[metric], color=COL[metric], lw=1, alpha=0.65,
            label=metric + " daily mean")
    ax.scatter(daily.index[daily[flag]], daily[metric][daily[flag]],
               color="red", marker="x", s=45,
               label="IQR outliers: " + str(int(daily[flag].sum())))
    ax.axhline(hi, color="red", ls=":", lw=1, label="IQR upper fence")
    ax.axhline(lo, color="green", ls=":", lw=1, label="IQR lower fence")
    ax.set_ylabel(metric + " (ug/m3)")
    ax.legend(fontsize=8, ncol=3)
    ax.grid(alpha=0.25)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))

plt.tight_layout()
plt.savefig("step3_outliers_iqr.png", bbox_inches="tight")
plt.show()

print("Saved: step3_outliers_iqr.png")
print("PM10 IQR outliers:", int(daily["PM10_iqr"].sum()),
      "| z-score outliers:", int(daily["PM10_z"].sum()))
print("PM2.5 IQR outliers:", int(daily["PM25_iqr"].sum()),
      "| z-score outliers:", int(daily["PM25_z"].sum()))

daily["PM10_winsor"] = daily["PM10"].clip(lower=pm10_lo, upper=pm10_hi)
daily["PM25_winsor"] = daily["PM25"].clip(lower=pm25_lo, upper=pm25_hi)

# ================================================================
# 5. Air-quality calendar
# ================================================================

AQI_C = ["#27ae60", "#f39c12", "#e67e22", "#e74c3c"]
AQI_L = ["Good (<10)", "Moderate (10-15)", "Sensitive (15-25)", "Unhealthy (>25)"]

def classify_pm25(v):
    if v < 10:
        return 0
    if v < 15:
        return 1
    if v < 25:
        return 2
    return 3

daily["AQI"] = daily["PM25"].apply(classify_pm25)
start = daily.index.min()
cal = np.full((7, 53), np.nan)

for date, row in daily.iterrows():
    col = int((date - start).days // 7)
    row_index = int(date.dayofweek)
    if 0 <= col < 53:
        cal[row_index, col] = float(row["AQI"])

fig, ax = plt.subplots(figsize=(16, 3.5))
fig.suptitle("Air Quality Calendar - Daily PM2.5 Classes",
             fontsize=13, fontweight="bold")
cmap = ListedColormap(AQI_C)
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap.N)
ax.imshow(cal, cmap=cmap, norm=norm, aspect="auto", interpolation="none")
ax.set_yticks(range(7))
ax.set_yticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])

month_ticks = []
month_names = []
for mo in range(1, 13):
    tick = int((pd.Timestamp("2023-" + str(mo).zfill(2) + "-01") - start).days // 7)
    if 0 <= tick < 53:
        month_ticks.append(tick)
        month_names.append(month_labels[mo - 1])

ax.set_xticks(month_ticks)
ax.set_xticklabels(month_names)
ax.legend(handles=[mpatches.Patch(color=AQI_C[i], label=AQI_L[i]) for i in range(4)],
          loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.45),
          fontsize=8, frameon=False)

plt.tight_layout()
plt.savefig("step4_air_quality_calendar.png", bbox_inches="tight")
plt.show()
print("Saved: step4_air_quality_calendar.png")

# ================================================================
# 6. Spatial hotspots and regional risk
# Static plot instead of GIF: faster and presentation-friendly.
# ================================================================

risk = (
    clean.filter(F.col("lat").between(47, 55) & F.col("lon").between(6, 15))
    .groupBy("sensor_id", "lat", "lon")
    .agg(
        F.count("*").alias("n"),
        F.round(F.avg("P2"), 2).alias("PM25_mean"),
        F.round(F.stddev("P2"), 2).alias("PM25_std")
    )
    .filter(F.col("n") > 100)
    .orderBy(F.desc("PM25_mean"))
    .toPandas()
)

if risk.empty:
    print("No regional risk rows found after filtering.")
else:
    top15 = risk.head(15)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle("Regional PM2.5 Hotspots - Germany 2023",
                 fontsize=13, fontweight="bold")

    sc = axes[0].scatter(risk["lon"], risk["lat"], c=risk["PM25_mean"],
                         cmap="YlOrRd", s=22, alpha=0.75)
    axes[0].set_title("Sensor Map Colored by Annual PM2.5")
    axes[0].set_xlabel("Longitude")
    axes[0].set_ylabel("Latitude")
    axes[0].set_xlim(5.8, 15.2)
    axes[0].set_ylim(47.0, 55.5)
    axes[0].grid(alpha=0.25)
    fig.colorbar(sc, ax=axes[0], label="PM2.5 mean (ug/m3)")

    bar_colors = [
        AQI_C[3] if v > 25 else AQI_C[2] if v > 15 else AQI_C[1]
        for v in top15["PM25_mean"]
    ]
    axes[1].barh(range(len(top15)), top15["PM25_mean"], color=bar_colors, alpha=0.85)
    axes[1].axvline(WHO["PM25"], color="red", ls="--", lw=1.5, label="WHO limit")
    axes[1].set_yticks(range(len(top15)))
    axes[1].set_yticklabels([
        str(round(r.lat, 2)) + "N, " + str(round(r.lon, 2)) + "E"
        for r in top15.itertuples()
    ], fontsize=7)
    axes[1].set_xlabel("Annual mean PM2.5 (ug/m3)")
    axes[1].set_title("Top 15 Highest PM2.5 Locations")
    axes[1].invert_yaxis()
    axes[1].legend(fontsize=8)
    axes[1].grid(axis="x", alpha=0.25)

    plt.tight_layout()
    plt.savefig("step5_regional_hotspots.png", bbox_inches="tight")
    plt.show()
    print("Saved: step5_regional_hotspots.png")

# ================================================================
# 7. Linear trend values and trend plot
# ================================================================

def make_trend_stats(series):
    s = series.dropna()
    if len(s) < 2:
        raise ValueError("Need at least 2 daily rows to fit a trend line.")
    x = np.arange(len(s), dtype=float)
    y = s.values.astype(float)
    slope, intercept = np.polyfit(x, y, 1)
    fitted = intercept + slope * x
    total_var = np.sum((y - y.mean()) ** 2)
    r2 = 0.0 if total_var == 0 else 1 - np.sum((y - fitted) ** 2) / total_var
    return {
        "dates": s.index,
        "values": y,
        "fitted": fitted,
        "slope": float(slope),
        "r2": float(r2),
    }

stats = {
    "PM10": make_trend_stats(daily["PM10"]),
    "PM25": make_trend_stats(daily["PM25"]),
}

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
fig.suptitle("Linear Trend Lines for Daily PM Values - Germany 2023",
             fontsize=13, fontweight="bold")

for ax, metric in zip(axes, ["PM10", "PM25"]):
    st = stats[metric]
    ax.plot(st["dates"], st["values"], color=COL[metric], lw=1, alpha=0.55,
            label="Daily mean")
    ax.plot(st["dates"], st["fitted"], color="black", lw=2,
            label="Trend: slope=" + str(round(st["slope"], 4)) +
                  " ug/m3/day, R2=" + str(round(st["r2"], 2)))
    ax.axhline(WHO[metric], color="red", ls=":", lw=1.3,
               label="WHO limit")
    ax.set_ylabel(metric + " (ug/m3)")
    ax.legend(fontsize=8, ncol=3)
    ax.grid(alpha=0.25)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))

plt.tight_layout()
plt.savefig("step6_trend_lines.png", bbox_inches="tight")
plt.show()
print("Saved: step6_trend_lines.png")

# ================================================================
# 8. Final summary for presentation
# ================================================================

good_days = int((daily["AQI"] <= 1).sum())
bad_days = int((daily["AQI"] >= 2).sum())
trend_dir = "improving" if stats["PM25"]["slope"] < 0 else "worsening"

if risk.empty:
    hotspot_text = "No hotspot summary available after filtering."
else:
    exceed_count = int((risk["PM25_mean"] > WHO["PM25"]).sum())
    hotspot_text = (
        str(exceed_count) + "/" + str(len(risk)) +
        " sensor locations exceed the WHO annual PM2.5 limit"
    )

print("")
print("=" * 70)
print("CONCLUSIONS - Germany Air Quality 2023")
print("=" * 70)
print("1. PM2.5 trend is " + trend_dir +
      " with slope " + str(round(stats["PM25"]["slope"], 4)) + " ug/m3/day.")
print("2. R2 is " + str(round(stats["PM25"]["r2"], 3)) +
      ", so the straight-line trend explains only part of the variation.")
print("3. Seasonal plots show hour, weekday, and monthly pollution patterns.")
print("4. IQR outlier detection found " + str(int(daily["PM25_iqr"].sum())) +
      " PM2.5 outlier days; z-score found " + str(int(daily["PM25_z"].sum())) + ".")
print("5. Good/moderate PM2.5 days: " + str(good_days) + "/" + str(len(daily)) +
      "; at-risk days: " + str(bad_days) + "/" + str(len(daily)) + ".")
print("6. " + hotspot_text + ".")
print("7. Best policy window: winter and weekday peak hours, when pollution is highest.")
print("=" * 70)

# ================================================================
# 9. Actionable recommendation
# This is the useful "so what?" part after the analysis.
# ================================================================

worst_month_num = int(monthly.sort_values("PM25", ascending=False).iloc[0]["month"])
worst_month_name = month_labels[worst_month_num - 1]
worst_hour = int(hourly.sort_values("PM25", ascending=False).iloc[0]["hour"])
worst_weekday = dow.sort_values("PM25", ascending=False).iloc[0]["label"]
worst_day = daily.sort_values("PM25", ascending=False).index[0]
worst_day_value = float(daily.sort_values("PM25", ascending=False).iloc[0]["PM25"])

print("")
print("=" * 70)
print("ACTION PLAN - What We Can Do With This Analysis")
print("=" * 70)
print("1. Focus monitoring and warnings in " + worst_month_name +
      ", the month with the highest average PM2.5.")
print("2. Focus daily attention around hour " + str(worst_hour) +
      ":00, the highest average PM2.5 hour.")
print("3. Watch " + str(worst_weekday) +
      ", the weekday with the highest average PM2.5.")
print("4. Investigate the worst PM2.5 day: " + str(worst_day.date()) +
      " with average PM2.5 = " + str(round(worst_day_value, 2)) + " ug/m3.")
if not risk.empty:
    worst_sensor = risk.iloc[0]
    print("5. Prioritize the highest-risk sensor area: lat " +
          str(round(worst_sensor["lat"], 2)) + ", lon " +
          str(round(worst_sensor["lon"], 2)) + ", average PM2.5 = " +
          str(round(worst_sensor["PM25_mean"], 2)) + " ug/m3.")
else:
    print("5. No reliable hotspot sensor found after filtering.")
print("6. Practical actions: public warnings, traffic reduction during peak windows,")
print("   heating/emission checks in high-risk months, and sensor validation on")
print("   extreme outlier days.")
print("=" * 70)

# ================================================================
# 10. Presentation dashboard plot
# A cleaner final visual: not just separate boring charts.
# ================================================================

fig = plt.figure(figsize=(16, 10), facecolor="#f4f7fb")
gs = fig.add_gridspec(3, 4, height_ratios=[0.7, 2.2, 1.6], hspace=0.45, wspace=0.35)
fig.suptitle("Germany PM2.5 Risk Dashboard 2023",
             fontsize=18, fontweight="bold", y=0.98)

cards = [
    ("Worst month", worst_month_name),
    ("Worst hour", str(worst_hour) + ":00"),
    ("Worst day", str(worst_day.date())),
    ("PM2.5 trend", str(round(stats["PM25"]["slope"], 4)) + " ug/m3/day"),
]

for i, (title, value) in enumerate(cards):
    ax_card = fig.add_subplot(gs[0, i])
    ax_card.set_facecolor("white")
    ax_card.text(0.05, 0.68, title, fontsize=11, color="#52616f",
                 fontweight="bold", transform=ax_card.transAxes)
    ax_card.text(0.05, 0.24, value, fontsize=19, color="#17202a",
                 fontweight="bold", transform=ax_card.transAxes)
    ax_card.set_xticks([])
    ax_card.set_yticks([])
    for spine in ax_card.spines.values():
        spine.set_color("#d9e2ec")

ax_main = fig.add_subplot(gs[1, :3])
ax_main.set_facecolor("white")
ax_main.fill_between(daily.index, daily["PM25"], color="#bfdbfe", alpha=0.75,
                     label="Daily PM2.5")
ax_main.plot(daily.index, daily["PM25_7d"], color="#1d4ed8", lw=2.6,
             label="7-day moving average")
ax_main.plot(daily.index, daily["PM25_30d"], color="#111827", lw=2,
             ls="--", label="30-day moving average")
ax_main.axhline(WHO["PM25"], color="#dc2626", lw=1.6, ls=":",
                label="WHO limit")
ax_main.scatter([worst_day], [worst_day_value], s=95, color="#dc2626",
                edgecolor="white", linewidth=1.5, zorder=5)
ax_main.annotate("Worst day\n" + str(round(worst_day_value, 2)) + " ug/m3",
                 xy=(worst_day, worst_day_value),
                 xytext=(15, 22), textcoords="offset points",
                 arrowprops=dict(arrowstyle="->", color="#dc2626"),
                 fontsize=10, fontweight="bold")
ax_main.set_title("Daily PM2.5 with Risk Threshold", fontweight="bold")
ax_main.set_ylabel("PM2.5 (ug/m3)")
ax_main.grid(alpha=0.22)
ax_main.legend(fontsize=8, ncol=4)
ax_main.xaxis.set_major_formatter(mdates.DateFormatter("%b"))

ax_month = fig.add_subplot(gs[1, 3])
ax_month.set_facecolor("white")
month_plot = monthly.copy()
month_plot["label"] = [month_labels[int(m) - 1] for m in month_plot["month"]]
month_colors = [
    "#dc2626" if v > 25 else "#f97316" if v > 15 else "#2563eb"
    for v in month_plot["PM25"]
]
ax_month.barh(month_plot["label"], month_plot["PM25"], color=month_colors, alpha=0.9)
ax_month.axvline(WHO["PM25"], color="#dc2626", ls=":", lw=1.6)
ax_month.set_title("Monthly PM2.5 Risk", fontweight="bold")
ax_month.set_xlabel("ug/m3")
ax_month.grid(axis="x", alpha=0.22)
ax_month.invert_yaxis()

ax_hour = fig.add_subplot(gs[2, :2])
ax_hour.set_facecolor("white")
ax_hour.plot(hourly["hour"], hourly["PM25"], color="#2563eb", lw=2.5, marker="o")
ax_hour.axvline(worst_hour, color="#dc2626", ls="--", lw=1.6,
                label="Worst hour")
ax_hour.fill_between(hourly["hour"], hourly["PM25"], color="#bfdbfe", alpha=0.65)
ax_hour.set_title("Average PM2.5 by Hour", fontweight="bold")
ax_hour.set_xlabel("Hour of day")
ax_hour.set_ylabel("PM2.5 (ug/m3)")
ax_hour.set_xticks(range(0, 24, 3))
ax_hour.grid(alpha=0.22)
ax_hour.legend(fontsize=8)

ax_map = fig.add_subplot(gs[2, 2:])
ax_map.set_facecolor("white")
if risk.empty:
    ax_map.text(0.5, 0.5, "No reliable hotspot data after filtering",
                ha="center", va="center", transform=ax_map.transAxes,
                fontsize=12, fontweight="bold")
else:
    sc = ax_map.scatter(risk["lon"], risk["lat"], c=risk["PM25_mean"],
                        cmap="YlOrRd", s=35, alpha=0.8,
                        edgecolor="white", linewidth=0.25)
    worst_sensor = risk.iloc[0]
    ax_map.scatter([worst_sensor["lon"]], [worst_sensor["lat"]],
                   s=140, marker="*", color="#1f2937",
                   edgecolor="white", linewidth=1.0, label="Highest-risk sensor")
    ax_map.set_xlim(5.8, 15.2)
    ax_map.set_ylim(47.0, 55.5)
    fig.colorbar(sc, ax=ax_map, label="PM2.5 mean (ug/m3)")
    ax_map.legend(fontsize=8, loc="upper right")
ax_map.set_title("Spatial Hotspots", fontweight="bold")
ax_map.set_xlabel("Longitude")
ax_map.set_ylabel("Latitude")
ax_map.grid(alpha=0.22)

plt.savefig("step7_presentation_dashboard.png", bbox_inches="tight")
plt.show()
print("Saved: step7_presentation_dashboard.png")


In [ ]:
%pyspark
# Dynamic presentation dashboard.
# Run the main analysis first, then paste/run this as a NEW Zeppelin paragraph.

import json
import pandas as pd

daily_dyn = daily.reset_index().copy()
daily_dyn["date"] = pd.to_datetime(daily_dyn["date"]).dt.strftime("%Y-%m-%d")

payload = {
    "daily": {
        "date": daily_dyn["date"].tolist(),
        "PM10": [round(float(x), 3) for x in daily_dyn["PM10"].tolist()],
        "PM25": [round(float(x), 3) for x in daily_dyn["PM25"].tolist()],
        "PM10_7d": [None if pd.isna(x) else round(float(x), 3) for x in daily_dyn["PM10_7d"].tolist()],
        "PM25_7d": [None if pd.isna(x) else round(float(x), 3) for x in daily_dyn["PM25_7d"].tolist()],
    },
    "hourly": {
        "hour": [int(x) for x in hourly["hour"].tolist()],
        "PM10": [round(float(x), 3) for x in hourly["PM10"].tolist()],
        "PM25": [round(float(x), 3) for x in hourly["PM25"].tolist()],
    },
    "monthly": {
        "month": [month_labels[int(x) - 1] for x in monthly["month"].tolist()],
        "PM10": [round(float(x), 3) for x in monthly["PM10"].tolist()],
        "PM25": [round(float(x), 3) for x in monthly["PM25"].tolist()],
    },
    "summary": {
        "worst_month": worst_month_name,
        "worst_hour": int(worst_hour),
        "worst_weekday": str(worst_weekday),
        "worst_day": str(worst_day.date()),
        "worst_day_value": round(float(worst_day_value), 2),
        "pm25_slope": round(float(stats["PM25"]["slope"]), 4),
        "pm25_r2": round(float(stats["PM25"]["r2"]), 3),
    }
}

html_template = r"""
%html
<div id="sapDash" class="sap-wrap">
  <style>
    .sap-wrap {
      font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
      color: #17202a;
      max-width: 1180px;
      margin: 0 auto;
      background: #f7f9fb;
      border: 1px solid #d9e2ec;
      padding: 18px;
    }
    .sap-head {
      display: flex;
      justify-content: space-between;
      gap: 16px;
      align-items: flex-start;
      margin-bottom: 14px;
    }
    .sap-title {
      font-size: 24px;
      font-weight: 800;
      margin: 0;
    }
    .sap-sub {
      margin: 4px 0 0;
      font-size: 13px;
      color: #52616f;
    }
    .sap-controls {
      display: flex;
      gap: 8px;
      align-items: center;
      flex-wrap: wrap;
    }
    .sap-btn {
      border: 1px solid #bcccdc;
      background: white;
      color: #1f2933;
      padding: 7px 10px;
      cursor: pointer;
      font-weight: 700;
    }
    .sap-btn.active {
      background: #1f6feb;
      border-color: #1f6feb;
      color: white;
    }
    .sap-grid {
      display: grid;
      grid-template-columns: repeat(4, 1fr);
      gap: 10px;
      margin-bottom: 14px;
    }
    .sap-card {
      background: white;
      border: 1px solid #d9e2ec;
      padding: 12px;
    }
    .sap-label {
      color: #627d98;
      font-size: 12px;
      font-weight: 700;
      text-transform: uppercase;
    }
    .sap-value {
      font-size: 22px;
      font-weight: 800;
      margin-top: 4px;
    }
    .sap-panel {
      background: white;
      border: 1px solid #d9e2ec;
      padding: 12px;
      margin-bottom: 12px;
    }
    .sap-panel h3 {
      margin: 0 0 8px;
      font-size: 15px;
    }
    .sap-slider {
      width: 100%;
      margin: 10px 0 4px;
    }
    .sap-note {
      font-size: 12px;
      color: #627d98;
      margin-top: 8px;
    }
    .sap-two {
      display: grid;
      grid-template-columns: 1fr 1fr;
      gap: 12px;
    }
    svg {
      width: 100%;
      height: auto;
      display: block;
      background: #ffffff;
    }
    .axis {
      stroke: #9fb3c8;
      stroke-width: 1;
    }
    .gridline {
      stroke: #e6eef5;
      stroke-width: 1;
    }
    .line-main {
      fill: none;
      stroke-width: 2.5;
    }
    .line-smooth {
      fill: none;
      stroke: #111827;
      stroke-width: 2;
      stroke-dasharray: 5 5;
    }
    .bar {
      opacity: 0.85;
    }
    @media (max-width: 850px) {
      .sap-grid, .sap-two {
        grid-template-columns: 1fr;
      }
      .sap-head {
        flex-direction: column;
      }
    }
  </style>

  <div class="sap-head">
    <div>
      <h2 class="sap-title">Germany Air Pollution 2023 - Dynamic Dashboard</h2>
      <p class="sap-sub">Use this for presentation: play the timeline, switch pollutant, then explain the action plan.</p>
    </div>
    <div class="sap-controls">
      <button class="sap-btn active" id="pm25Btn">PM2.5</button>
      <button class="sap-btn" id="pm10Btn">PM10</button>
      <button class="sap-btn" id="playBtn">Play</button>
    </div>
  </div>

  <div class="sap-grid">
    <div class="sap-card">
      <div class="sap-label">Worst Month</div>
      <div class="sap-value" id="worstMonth"></div>
    </div>
    <div class="sap-card">
      <div class="sap-label">Worst Hour</div>
      <div class="sap-value" id="worstHour"></div>
    </div>
    <div class="sap-card">
      <div class="sap-label">Worst Day</div>
      <div class="sap-value" id="worstDay"></div>
    </div>
    <div class="sap-card">
      <div class="sap-label">PM2.5 Trend</div>
      <div class="sap-value" id="trendValue"></div>
    </div>
  </div>

  <div class="sap-panel">
    <h3 id="timelineTitle">Daily PM2.5 Timeline</h3>
    <svg id="timeline" viewBox="0 0 1100 420" role="img"></svg>
    <input id="daySlider" class="sap-slider" type="range" min="5" value="60">
    <div class="sap-note" id="dayReadout"></div>
  </div>

  <div class="sap-two">
    <div class="sap-panel">
      <h3 id="hourTitle">Average by Hour</h3>
      <svg id="hourChart" viewBox="0 0 540 330" role="img"></svg>
    </div>
    <div class="sap-panel">
      <h3 id="monthTitle">Average by Month</h3>
      <svg id="monthChart" viewBox="0 0 540 330" role="img"></svg>
    </div>
  </div>
</div>

<script>
(function() {
  const data = __DATA__;
  const color = { PM25: "#2563eb", PM10: "#dc493a" };
  const who = { PM25: 15, PM10: 45 };
  let metric = "PM25";
  let playing = false;
  let timer = null;

  const slider = document.getElementById("daySlider");
  slider.max = data.daily.date.length - 1;
  document.getElementById("worstMonth").textContent = data.summary.worst_month;
  document.getElementById("worstHour").textContent = data.summary.worst_hour + ":00";
  document.getElementById("worstDay").textContent = data.summary.worst_day;
  document.getElementById("trendValue").textContent =
    data.summary.pm25_slope + " /day";

  function clear(svg) {
    while (svg.firstChild) svg.removeChild(svg.firstChild);
  }

  function el(name, attrs) {
    const n = document.createElementNS("http://www.w3.org/2000/svg", name);
    Object.keys(attrs || {}).forEach(k => n.setAttribute(k, attrs[k]));
    return n;
  }

  function scale(v, inMin, inMax, outMin, outMax) {
    if (inMax === inMin) return (outMin + outMax) / 2;
    return outMin + (v - inMin) * (outMax - outMin) / (inMax - inMin);
  }

  function linePath(values, maxIndex, x0, y0, w, h, yMax) {
    let d = "";
    for (let i = 0; i <= maxIndex; i++) {
      const v = values[i];
      if (v === null || v === undefined) continue;
      const x = scale(i, 0, values.length - 1, x0, x0 + w);
      const y = scale(v, 0, yMax, y0 + h, y0);
      d += (d ? " L " : "M ") + x.toFixed(1) + " " + y.toFixed(1);
    }
    return d;
  }

  function drawTimeline() {
    const svg = document.getElementById("timeline");
    clear(svg);
    const values = data.daily[metric];
    const smooth = data.daily[metric + "_7d"];
    const maxIndex = Number(slider.value);
    const margin = { left: 62, right: 24, top: 22, bottom: 46 };
    const w = 1100 - margin.left - margin.right;
    const h = 420 - margin.top - margin.bottom;
    const yMax = Math.max(who[metric] * 1.18, Math.max(...values) * 1.12);

    for (let g = 0; g <= 5; g++) {
      const y = margin.top + (h * g / 5);
      svg.appendChild(el("line", { x1: margin.left, y1: y, x2: margin.left + w, y2: y, class: "gridline" }));
      const val = Math.round(scale(y, margin.top + h, margin.top, 0, yMax));
      const t = el("text", { x: 12, y: y + 4, "font-size": "12", fill: "#52616f" });
      t.textContent = val;
      svg.appendChild(t);
    }

    const whoY = scale(who[metric], 0, yMax, margin.top + h, margin.top);
    svg.appendChild(el("line", {
      x1: margin.left, y1: whoY, x2: margin.left + w, y2: whoY,
      stroke: "#ef4444", "stroke-width": 2, "stroke-dasharray": "4 4"
    }));
    const whoText = el("text", { x: margin.left + w - 120, y: whoY - 7, "font-size": "12", fill: "#ef4444" });
    whoText.textContent = "WHO " + who[metric];
    svg.appendChild(whoText);

    svg.appendChild(el("path", {
      d: linePath(values, maxIndex, margin.left, margin.top, w, h, yMax),
      class: "line-main",
      stroke: color[metric]
    }));
    svg.appendChild(el("path", {
      d: linePath(smooth, maxIndex, margin.left, margin.top, w, h, yMax),
      class: "line-smooth"
    }));

    const cx = scale(maxIndex, 0, values.length - 1, margin.left, margin.left + w);
    const cy = scale(values[maxIndex], 0, yMax, margin.top + h, margin.top);
    svg.appendChild(el("circle", { cx: cx, cy: cy, r: 6, fill: color[metric], stroke: "white", "stroke-width": 2 }));
    const label = el("text", { x: Math.min(cx + 10, margin.left + w - 160), y: Math.max(cy - 12, margin.top + 15), "font-size": "13", fill: "#17202a", "font-weight": "700" });
    label.textContent = data.daily.date[maxIndex] + "  " + values[maxIndex] + " ug/m3";
    svg.appendChild(label);

    svg.appendChild(el("line", { x1: margin.left, y1: margin.top + h, x2: margin.left + w, y2: margin.top + h, class: "axis" }));
    svg.appendChild(el("line", { x1: margin.left, y1: margin.top, x2: margin.left, y2: margin.top + h, class: "axis" }));

    document.getElementById("timelineTitle").textContent = "Daily " + metric + " Timeline";
    document.getElementById("dayReadout").textContent =
      "Showing up to " + data.daily.date[maxIndex] + ". Blue/red line = daily mean, dashed line = 7-day moving average.";
  }

  function drawBarChart(svgId, labels, values, titleId, titleText) {
    const svg = document.getElementById(svgId);
    clear(svg);
    document.getElementById(titleId).textContent = titleText;
    const margin = { left: 48, right: 16, top: 18, bottom: 54 };
    const w = 540 - margin.left - margin.right;
    const h = 330 - margin.top - margin.bottom;
    const maxVal = Math.max(who[metric], Math.max(...values)) * 1.2;
    const bw = w / values.length * 0.72;

    for (let g = 0; g <= 4; g++) {
      const y = margin.top + h * g / 4;
      svg.appendChild(el("line", { x1: margin.left, y1: y, x2: margin.left + w, y2: y, class: "gridline" }));
    }
    const whoY = scale(who[metric], 0, maxVal, margin.top + h, margin.top);
    svg.appendChild(el("line", { x1: margin.left, y1: whoY, x2: margin.left + w, y2: whoY, stroke: "#ef4444", "stroke-width": 2, "stroke-dasharray": "4 4" }));

    values.forEach((v, i) => {
      const x = margin.left + i * (w / values.length) + (w / values.length - bw) / 2;
      const y = scale(v, 0, maxVal, margin.top + h, margin.top);
      svg.appendChild(el("rect", { x: x, y: y, width: bw, height: margin.top + h - y, fill: color[metric], class: "bar" }));
      const t = el("text", { x: x + bw / 2, y: 304, "text-anchor": "middle", "font-size": labels.length > 12 ? "9" : "11", fill: "#52616f" });
      t.textContent = labels[i];
      svg.appendChild(t);
    });

    svg.appendChild(el("line", { x1: margin.left, y1: margin.top + h, x2: margin.left + w, y2: margin.top + h, class: "axis" }));
    svg.appendChild(el("line", { x1: margin.left, y1: margin.top, x2: margin.left, y2: margin.top + h, class: "axis" }));
  }

  function redraw() {
    document.getElementById("pm25Btn").classList.toggle("active", metric === "PM25");
    document.getElementById("pm10Btn").classList.toggle("active", metric === "PM10");
    drawTimeline();
    drawBarChart("hourChart", data.hourly.hour.map(h => String(h)), data.hourly[metric],
                 "hourTitle", "Average " + metric + " by Hour");
    drawBarChart("monthChart", data.monthly.month, data.monthly[metric],
                 "monthTitle", "Average " + metric + " by Month");
  }

  document.getElementById("pm25Btn").onclick = () => { metric = "PM25"; redraw(); };
  document.getElementById("pm10Btn").onclick = () => { metric = "PM10"; redraw(); };
  slider.oninput = drawTimeline;
  document.getElementById("playBtn").onclick = () => {
    playing = !playing;
    document.getElementById("playBtn").textContent = playing ? "Pause" : "Play";
    if (playing) {
      timer = setInterval(() => {
        let v = Number(slider.value) + 2;
        if (v >= data.daily.date.length) v = 5;
        slider.value = v;
        drawTimeline();
      }, 90);
    } else {
      clearInterval(timer);
    }
  };

  redraw();
})();
</script>
"""

print(html_template.replace("__DATA__", json.dumps(payload)))
